<a href="https://colab.research.google.com/github/metalBigLesson/PySolitaire/blob/main/Solitaire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from enum import IntEnum
import random
import traceback

class Suit(IntEnum):
    NONE = 0
    CLUBS = 1
    DIAMONDS = 2
    SPADES = 3
    HEARTS = 4

class Color(IntEnum):
    NONE = 0
    RED = -1
    BLACK = 1

def color(suit: Suit):
    if (suit == Suit.NONE):
        return Color.NONE
    if (suit % 2 == 0): return Color.RED
    return Color.BLACK

class Card:
    def __init__(self, suit: Suit, rank: int, show=False):
        self.suit = suit
        self.rank = rank
        self.show = show

    def __eq__(self, other: "Card"):
        return self.suit == other.suit and self.rank == other.rank

    def __sum__(self, other: "Card"):
        return self.rank + other.rank

    def __sub__(self, other: "Card"):
        return self.rank - other.rank

    def __mul__(self, other: "Card"):
        return color(self.suit) * color(other.suit)

    def __mod__(self, other: "Card"):
        return self.suit - other.suit

    def __str__(self):
        if not self.show:
            return "[▒▒]"
        return '[' + self._suit_chr() + self._rank_str() + ']'

    def _suit_chr(self):
        match self.suit:
            case Suit.CLUBS:
                return "♣"
            case Suit.DIAMONDS:
                return "◇"
            case Suit.HEARTS:
                return "♡"
            case Suit.SPADES:
                return "♠"
            case _:
                return "　"

    def _rank_str(self):
        match self.rank:
            case 0:
                return "  "
            case 1:
                return "Ａ"
            case 10:
                return "10"
            case 11:
                return "Ｊ"
            case 12:
                return "Ｑ"
            case 13:
                return "Ｋ"
            case _:
                return chr(ord('０') + self.rank)

class Hand():
    def __init__(self):
        self.selected_ref: list[Card] = []
        self.max_length = 0

    def __str__(self):
        selected_str = ' '
        for card in self.selected_ref:
            selected_str += card.__str__()[:-1]
        if selected_str != ' ':
            selected_str += ']'
        return selected_str

    def bottom(self):
        if (len(self.selected_ref) == 0):
            return Card(Suit.NONE, 0)
        return self.selected_ref[0]

    def length(self):
        return len(self.selected_ref)

    def select(self, new_ref: list[Card]):
        self.selected_ref = new_ref

    def move(self):
        selected = self.selected_ref[:]
        self.selected_ref.clear()
        return selected

    def can_select(self, count: int):
        return self.max_length == 0 or self.max_length >=count

class Card_Structure():
    def __init__(self, card_list: list[Card], can_move_multiple: bool, can_pop_some: bool):
        self.cards = card_list
        self.selected: list[Card] = []
        self.can_move_multiple = can_move_multiple
        self.can_pop_some = can_pop_some

    def is_legal(self, card1: Card, card2: Card):
        return True

    def top(self):
        if (len(self.cards) == 0):
            return Card(Suit.NONE, 0)
        return self.cards[-1]

    def length(self):
        return len(self.cards)

    def can_push(self, hand: Hand):
        if hand.selected_ref is self.selected: return True
        if hand.length() == 0: return False
        if not self.can_move_multiple and hand.length() > 1: return False
        return self.is_legal(self.top(), hand.bottom())

    def can_pop(self, index: int, hand: Hand):
        if index < 0 or index >= self.length(): return False
        if not self.can_pop_some or hand.length() > 0 or not hand.can_select(self.length() - index): return False
        if not self.can_move_multiple and index != self.length() - 1: return False
        for i in range(index,  self.length() - 1):
            if not self.is_legal(self.cards[i], self.cards[i+1]):
                return False
        return True

    def push_if_can(self, hand: Hand):
        if self.can_push(hand):
            self.cards.extend(hand.move())

    def pop_if_can(self, index: int, hand: Hand):
        if self.can_pop(index, hand):
            self.selected = self.cards[index:]
            hand.select(self.selected)
            self.cards = self.cards[:index]

    def on_click(self, index: int, hand: Hand):
        if hand.length() == 0:
            #print('try_pop')
            self.pop_if_can(index, hand)
        else:
            #print('try_push')
            self.push_if_can(hand)

    def open(self):
        pass


def make_cards(is_shuffle = True):
    deck: list[Card] = []
    for suit in Suit:
        if suit == Suit.NONE:
            continue
        for rank in range(1, 13):
            deck.append(Card(suit, rank))
    if is_shuffle:
        random.shuffle(deck)
    return deck

In [ ]:
class Tableau(Card_Structure):
    def __init__(self, card_list: list[Card]):
        super().__init__(card_list, True, True)

    def __str__(self):
        stack_str = ''
        for card in self.cards:
            stack_str += card.__str__()[:-1]
        if stack_str != '':
            stack_str += ']'
        selected_str = ' '
        for card in self.selected:
            selected_str += card.__str__()[:-1]
        if selected_str != ' ':
            selected_str += ']'
        return stack_str + selected_str

class Klondike_Tableau(Tableau):
    def __init__(self, card_list: list[Card]):
        super().__init__(card_list)

    def is_legal(self, card1, card2):
        if not card1.show:
            return False
        if card1.suit == Suit.NONE:
            return card2.rank == 13
        return card1 * card2 == -1 and card1 - card2 == 1

    def open(self):
        if len(self.cards) != 0 and len(self.selected) == 0:
            self.cards[-1].show = True

class Foundation(Card_Structure):
    def __init__(self, can_pop_some):
        super().__init__([], False, can_pop_some)
        self.foundations = [[Card(Suit.CLUBS, 0, True)],[Card(Suit.DIAMONDS, 0, True)],[Card(Suit.HEARTS, 0, True)],[Card(Suit.SPADES, 0, True)]]

    def __str__(self):
        string = '  '
        if len(self.selected) != 0:
            string += '      ' * int(self.selected[0].suit - 1)
            string += self.selected[0].__str__()
        string += '\nF '
        for foundation in self.foundations:
            string += foundation[-1].__str__()
        return string

    def top(self, index: int):
        return self.foundations[index][-1]

    def push_if_can(self, hand):
        if hand.length() != 1: return
        index = int(hand.bottom().suit) - 1
        if hand.bottom() - self.top(index) == 1:
            self.foundations[index].extend(hand.move())

    def pop_if_can(self, index, hand):
        if not self.can_pop_some: return
        self.selected = [self.foundations[index].pop()]
        hand.select(self.selected)

class Waste(Card_Structure):
    def __init__(self):
        super().__init__([], False, False)

    def __str__(self):
        stack_str = ''
        is_selected = len(self.selected) > 0
        show_count = 2 if is_selected else 3
        visibles = self.cards[-show_count:]
        for card in visibles:
            stack_str += card.__str__()[:-1]
        if stack_str != '':
            stack_str += ']'
        selected_str = ' '
        if is_selected:
            selected_str += self.selected[0].__str__()
        return stack_str + selected_str


In [ ]:
class Game():

    def __init__(self):
        self.deck = make_cards()
        self.stock: list[Card] = []
        self.layout: dict[str, Card_Structure] = {}
        self.setup()
        self.hand = Hand()

    def setup(self):
        pass

    def open_stock(self):
        pass

    def print_game(self):
        print('not a valid game')
        print('Q: quit')

    def play(self):
        while True:
            self.print_game()
            try:
                input_str = input()
                if (input_str == 'Q'):
                    break
                first_char = input_str[0]
                rear_str = input_str[1:]
                index = int(rear_str) if rear_str != '' else 0
                if first_char == 'o':
                    self.open_stock()
                else:
                    self.layout[first_char].on_click(index, self.hand)
            except Exception as e:
                print('wrong input')
                #traceback.print_exc()
                continue

In [ ]:
class Klondike(Game):
    def __init__(self, opening_count: int):
        super().__init__()
        self.opening_count = opening_count

    def setup(self):
        self.layout['w'] = Waste()
        self.layout['f'] = Foundation(True)
        for i in range(7):
            temp_cards = []
            for j in range(i + 1):
                temp_cards.append(self.deck.pop())
            self.layout[str(i)] = Klondike_Tableau(temp_cards)
        self.stock = self.deck

    def open_stock(self):
        if len(self.stock) == 0:
            for i in range(len(self.layout['w'].cards)):
                card = self.layout['w'].cards.pop()
                card.show = False
                self.stock.append(card)
            return
        for i in range(self.opening_count):
            if len(self.stock) == 0: break
            card = self.stock.pop()
            card.show = True
            self.layout['w'].cards.append(card)

    def print_game(self):
        print(self.layout['f'])
        stock_str = '[▒▒]' if len(self.stock) != 0 else '  ○  '
        print('W', stock_str, self.layout['w'])
        for i in range(7):
            self.layout[str(i)].open()
            print(i, ' ', self.layout[str(i)], sep='')
        print('  ０   １   ２   ３   ４   ５   ６   ７   ８   ９   10   11   12')
        #selected_str = 'None' if len(self.hand.selected_ref) == 0 else self.hand.selected_ref[0]
        #print('selected card: ', selected_str)
        opening_str = 'a card' if self.opening_count == 1 else str(self.opening_count) + ' cards'
        print('Q: quit')
        print('o: turn', opening_str, 'from the stock to the waste')
        print('w: select a top card from the waste')
        print('f(col): select a card from the foundation')
        print('(row)(col): select (a) card(s) from the tableau')
        print('f: stack the selected card to the foundation')
        print('(row): move the selected card(s) to the tableau')

In [ ]:
print('Select a game')
print('1: Klondike (easy)')
print('2: Klondike (hard)')
game_no = input()
if game_no == '1':
    klondike = Klondike(1)
    klondike.play()
elif game_no == '2':
    klondike = Klondike(3)
    klondike.play()

Select a game
1: Klondike (easy)
2: Klondike (hard)
1
  
F [♣  ][◇  ][♡  ][♠  ]
W [▒▒]  
0 [♡３] 
1 [▒▒[♠５] 
2 [▒▒[▒▒[♣５] 
3 [▒▒[▒▒[▒▒[♣４] 
4 [▒▒[▒▒[▒▒[▒▒[♣７] 
5 [▒▒[▒▒[▒▒[▒▒[▒▒[◇３] 
6 [▒▒[▒▒[▒▒[▒▒[▒▒[▒▒[♡９] 
  ０   １   ２   ３   ４   ５   ６   ７   ８   ９   10   11   12
Q: quit
o: turn a card from the stock to the waste
w: select a top card from the waste
f(col): select a card from the foundation
(row)(col): select (a) card(s) from the tableau
f: stack the selected card to the foundation
(row): move the selected card(s) to the tableau
o
  
F [♣  ][◇  ][♡  ][♠  ]
W [▒▒] [♡６] 
0 [♡３] 
1 [▒▒[♠５] 
2 [▒▒[▒▒[♣５] 
3 [▒▒[▒▒[▒▒[♣４] 
4 [▒▒[▒▒[▒▒[▒▒[♣７] 
5 [▒▒[▒▒[▒▒[▒▒[▒▒[◇３] 
6 [▒▒[▒▒[▒▒[▒▒[▒▒[▒▒[♡９] 
  ０   １   ２   ３   ４   ５   ６   ７   ８   ９   10   11   12
Q: quit
o: turn a card from the stock to the waste
w: select a top card from the waste
f(col): select a card from the foundation
(row)(col): select (a) card(s) from the tableau
f: stack the selected card to the foundation
(row): move the selected card(

KeyboardInterrupt: Interrupted by user

D [▒▒][♣Ｋ[♠Ｋ[♡２]

H [♡  ][◇  ][♣  ][♠  ]

0 [♠Ｊ]

1 [▒▒[♣５]

2 [▒▒[▒▒[♠Ｑ]

3 [▒▒[▒▒[▒▒[◇Ａ]

4 [▒▒[▒▒[▒▒[▒▒[♠10]

5 [▒▒[▒▒[▒▒[▒▒[▒▒] [◇３]

6 [▒▒[▒▒[▒▒[▒▒[▒▒[▒▒[♣４]

　0　　1　　2　　3　　4　　5　　6